# DocuVision AI — Training Notebook (run on Colab, free T4 GPU)

Runtime > Change runtime type > **T4 GPU** before running this.

In [ ]:
!nvidia-smi

In [1]:
from typing_extensions import Doc
%cd /content
!rm -rf DocuVisionAI
!git clone https://github.com/fathimarfa/DocuVisionAI.git
%cd DocuVisionAI
!cat requirements.txt

/content
Cloning into 'DocuVisionAI'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 60 (delta 19), reused 49 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 20.70 KiB | 6.90 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/DocuVisionAI
transformers>=4.44.0
datasets>=2.19.0
Pillow>=10.1.0
jiwer==3.0.0
accelerate>=0.24.0
tensorboard>=2.15.0
PyYAML>=6.0.1
tqdm>=4.66.1
pandas>=2.1.3
numpy>=1.26.2
scikit-learn>=1.3.2
pytest>=7.4.3

In [2]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1,000.0/1,000.0 kB 54.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# 2. Download + split the dataset
!python data/download_dataset.py --config configs/trocr_base.yaml

2026-07-20 17:38:07,959 [INFO] Downloading dataset: Teklia/IAM-line
2026-07-20 17:38:08,248 [INFO] HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-07-20 17:38:08,490 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/Teklia/IAM-line/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-07-20 17:38:08,491 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-20 17:38:08,498 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Teklia/IAM-line/fbdad97500ce54635c0d1ba306bf535cb40656cf/README.md "HTTP/1.1 200 OK"
2026-07-20 17:38:08,510 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/Teklia/IAM-line/fbdad97500ce54635c0d1ba306bf535cb40656cf/README.md "HTTP/1.1 200 OK"
README.md: 100% 2.14k/2.14k [00:00<00:00, 7.94MB/s]
2026-07-20 17:38:08,778 [INFO] HTTP Request: HEAD https://huggingface.co/d

In [6]:
# 3. Fine-tune TrOCR (mixed precision + gradient accumulation configured in YAML)
!python -m src.train --config configs/trocr_base.yaml

2026-07-20 17:47:59,589 [INFO] HTTP Request: GET https://huggingface.co/api/models/microsoft/trocr-base-handwritten/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-20 17:47:59,832 [INFO] HTTP Request: HEAD https://huggingface.co/microsoft/trocr-base-handwritten/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 17:47:59,832 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-20 17:48:00,061 [INFO] HTTP Request: HEAD https://huggingface.co/microsoft/trocr-base-handwritten/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-07-20 17:48:00,297 [INFO] HTTP Request: HEAD https://huggingface.co/microsoft/trocr-base-handwritten/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-07-20 17:48:00,532 [INFO] HTTP Request: HEAD https://huggingface.co/microsoft/trocr-base-handwritten/resolve/main/audio_toke

In [ ]:
# 4. Evaluate: fine-tuned vs. base model zero-shot (CER / WER report)
!python src/evaluate.py --checkpoint weights/checkpoint-best --base-model microsoft/trocr-base-handwritten

In [ ]:
# 5. Zip the best checkpoint and download it locally (for use in VS Code / app.py)
!zip -r checkpoint-best.zip weights/checkpoint-best
from google.colab import files
files.download('checkpoint-best.zip')

In [ ]:
# 6. (Optional) push the trained checkpoint + results back to your repo
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"
# !git add weights/checkpoint-best
# !git commit -m "Add trained checkpoint"
# !git push